In [2]:
from IPython.display import display, HTML
from IPython.core.magic import register_cell_magic
import uuid

display(HTML("""
<style>
.tutorial-note {
    padding: 12px 16px;
    margin: 12px 0;
    border-left: 5px solid #4c8bf5;
    background: rgba(76, 139, 245, 0.08);
    border-radius: 4px;
}

.tutorial-important {
    padding: 12px 16px;
    margin: 12px 0;
    border-left: 5px solid #e67e22;
    background: rgba(230, 126, 34, 0.08);
    border-radius: 4px;
}

.tutorial-warning {
    padding: 12px 16px;
    margin: 12px 0;
    border-left: 5px solid #d9534f;
    background: rgba(217, 83, 79, 0.08);
    border-radius: 4px;
}

.tutorial-result {
    padding: 12px 16px;
    margin: 12px 0;
    border-left: 5px solid #2ca02c;
    background: rgba(44, 160, 44, 0.08);
    border-radius: 4px;
}
</style>
"""))



@register_cell_magic
def important(line, cell):
    marker = f"important-{uuid.uuid4().hex}"

    display(HTML(f"""
    <div id="{marker}"></div>

    <style>
    .jp-CodeCell:has(#{marker}) {{
        background: rgba(230, 126, 34, 0.08);
        border-left: 5px solid #e67e22;
        border-radius: 5px;
        padding-left: 6px;
    }}
    </style>
    """))

    get_ipython().run_cell(cell)

In [3]:
import numpy as np
import scipy.sparse as sp

from scipy.sparse.linalg import splu
from scipy.optimize import minimize_scalar

from sqoct_core import (
    build_ysr_system,
    MIMP, I4
)

from sympy import Matrix, nsimplify
from IPython.display import display

In [8]:
def show_matrix(M):
    display(Matrix(np.asarray(M)).applyfunc(nsimplify))

<div class="tutorial-important">

Here the idea is to construct the local superconducting Green function at the impurity site and use its zero-energy pole condition to determine the YSR critical exchange $J_c$.

Then extract the corresponding normal-state spectrum and LDOS to connect the superconducting impurity response with the underlying electronic structure.
 
</div>

In [4]:
Nx = 2
Ny = 2

t1 = 1.0
t2 = 0.5

mu = 0.3
Delta = 0.2

fsyst, impurity_site, impurity_index = build_ysr_system(
    Nx, Ny,
    t1=t1, t2=t2,
    impurity_sublattice="A"
)

print("Impurity site =", impurity_site)
print("Impurity index =", impurity_index)

Impurity site = <Site [1 1] of sqo0>
Impurity index = 11


<div class="tutorial-note">
    
The retarded Green function is

$$
G_0(E)=
\left[(E+i\eta)I-H_0\right]^{-1},
$$

where $H_0$ is the **clean system**, so we set $J=0$.

We only need the $4\times4$ block corresponding to the impurity site.

</div>

In [17]:
E = 0.0
eta = 1e-8

H0 = fsyst.hamiltonian_submatrix(
    params={"J": 0.0, "mu": mu, "Delta": Delta},
    sparse=False
)

N = H0.shape[0]

G = np.linalg.inv((E + 1j * eta) * np.eye(N) - H0)

inds = 4 * impurity_index + np.arange(4)

Gii_dense = G[np.ix_(inds, inds)]  # the local Green function G_ii(E) in the four-component Nambu basis of the impurity site.

# Matrix(Gii_dense)
print(Gii)

[[-0.11497978-3.10438789e-08j  0.        +0.00000000e+00j
   0.        +0.00000000e+00j -0.62087758+2.48154184e-24j]
 [ 0.        +0.00000000e+00j -0.11497978-3.10438789e-08j
   0.62087758+0.00000000e+00j  0.        +0.00000000e+00j]
 [ 0.        +0.00000000e+00j  0.62087758+0.00000000e+00j
   0.11497978-3.10438789e-08j  0.        +0.00000000e+00j]
 [-0.62087758+3.30872245e-24j  0.        +0.00000000e+00j
   0.        +0.00000000e+00j  0.11497978-3.10438789e-08j]]


In [15]:
%%important

def local_clean_green(fsyst, impurity_index, mu, Delta, E=0.0, eta=1e-8):

    H0 = fsyst.hamiltonian_submatrix(
        sparse=True,
        params={"J": 0.0, "mu": mu, "Delta": Delta}
    ).tocsc()

    N = H0.shape[0]

    mat = (E + 1j * eta) * sp.eye(
        N, dtype=complex, format="csc"
    ) - H0

    lu = splu(mat)

    inds = 4 * impurity_index + np.arange(4)

    rhs = np.zeros((N, 4), dtype=complex)

    for a, ind in enumerate(inds):
        rhs[ind, a] = 1.0

    X = lu.solve(rhs)

    return X[inds, :]

In [16]:
Gii = local_clean_green(
    fsyst, impurity_index,
    mu=mu, Delta=Delta
)

print(Gii)

[[-0.11497978-3.10438789e-08j  0.        +0.00000000e+00j
   0.        +0.00000000e+00j -0.62087758+2.48154184e-24j]
 [ 0.        +0.00000000e+00j -0.11497978-3.10438789e-08j
   0.62087758+0.00000000e+00j  0.        +0.00000000e+00j]
 [ 0.        +0.00000000e+00j  0.62087758+0.00000000e+00j
   0.11497978-3.10438789e-08j  0.        +0.00000000e+00j]
 [-0.62087758+3.30872245e-24j  0.        +0.00000000e+00j
   0.        +0.00000000e+00j  0.11497978-3.10438789e-08j]]


### YSR pole strength
<div class="tutorial-note">
For a magnetic impurity, $V_{\rm imp}=J M_{\rm imp}$. The impurity pole condition is

$$
\det\left[
I-JG_{ii}(E)M_{\rm imp}
\right]=0.
$$

At the zero-energy transition, set $E=0$.
</div>

In [19]:
J = 0.5

D = I4 - J * (Gii @ MIMP)

print(D)

[[1.05748989+1.55219394e-08j 0.        +0.00000000e+00j
  0.        +0.00000000e+00j 0.31043879-1.24077092e-24j]
 [0.        +0.00000000e+00j 0.94251011-1.55219394e-08j
  0.31043879+0.00000000e+00j 0.        +0.00000000e+00j]
 [0.        +0.00000000e+00j 0.31043879+0.00000000e+00j
  1.05748989-1.55219394e-08j 0.        +0.00000000e+00j]
 [0.31043879-1.65436123e-24j 0.        +0.00000000e+00j
  0.        +0.00000000e+00j 0.94251011+1.55219394e-08j]]


In [22]:
svals = np.linalg.svd(D,compute_uv=False)

print(svals)
print("Smallest singular value =", svals[-1])

[1.31571717 1.31571717 0.68428283 0.68428283]
Smallest singular value = 0.6842828337981008


In [23]:
%%important

def ysr_pole_strength(J, Gii):

    D = I4 - J * (Gii @ MIMP)

    return np.linalg.svd(
        D,
        compute_uv=False
    )[-1]

Now we want to find $J_c$ from the Green function

In [24]:
J_values = np.linspace(0.0, 2.0, 401)

pole_values = np.array([
    ysr_pole_strength(J, Gii)
    for J in J_values
])

index = np.argmin(pole_values)

print("Approximate Jc =", J_values[index])
print("Minimum =", pole_values[index])

Approximate Jc = 1.585
Minimum = 0.0008234168614906267


Once we have approximate minima, we can refine around that point.

In [25]:
i = np.argmin(pole_values)

Jlo = J_values[max(0, i - 1)]
Jhi = J_values[min(len(J_values) - 1, i + 1)]

result = minimize_scalar(
    lambda J: ysr_pole_strength(J, Gii),
    bounds=(Jlo, Jhi),
    method="bounded"
)

Jc = result.x

print("Jc =", Jc)
print("Pole minimum =", result.fun)

Jc = 1.5836970951022937
Pole minimum = 7.196570632455346e-07


In [26]:
%%important

def Jc_from_green(Gii, Jmax=2.0, nscan=401):

    J_scan = np.linspace(0.0, Jmax, nscan)

    pole = np.array([
        ysr_pole_strength(J, Gii)
        for J in J_scan
    ])

    i = np.argmin(pole)

    Jlo = J_scan[max(0, i - 1)]
    Jhi = J_scan[min(len(J_scan) - 1, i + 1)]

    result = minimize_scalar(
        lambda J: ysr_pole_strength(J, Gii),
        bounds=(Jlo, Jhi),
        method="bounded"
    )

    return result.x, result.fun

In [27]:
Jc, pole_min = Jc_from_green(
    Gii,
    Jmax=2.0
)

print("Jc =", Jc)
print("Pole minimum =", pole_min)

Jc = 1.5836970951022937
Pole minimum = 7.196570632455346e-07
